# Feature engineering och enkel modell\n
\n
Notebook som visar en enkel end-to-end pipeline: skapa features, traena modell och utvardera resultat.

In [ ]:
from pathlib import Path\n
\n
import numpy as np\n
import pandas as pd\n
from sklearn.compose import ColumnTransformer\n
from sklearn.impute import SimpleImputer\n
from sklearn.metrics import mean_absolute_error, r2_score\n
from sklearn.model_selection import train_test_split\n
from sklearn.pipeline import Pipeline\n
from sklearn.preprocessing import OneHotEncoder, StandardScaler\n
from sklearn.ensemble import RandomForestRegressor

In [ ]:
processed_dir = Path('../data/processed')\n
processed_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Skapa demo-data med tydligt mal (target)\n
rng = np.random.default_rng(7)\n
n = 500\n
\n
df = pd.DataFrame({\n
    'age': rng.integers(18, 70, size=n),\n
    'experience_years': rng.integers(0, 40, size=n),\n
    'department': rng.choice(['sales', 'tech', 'hr', 'ops'], size=n, p=[0.35, 0.3, 0.15, 0.2]),\n
    'city': rng.choice(['stockholm', 'goteborg', 'malmo'], size=n, p=[0.5, 0.3, 0.2]),\n
})\n
\n
# Target med brus: monthly_spend\n
base = 100 + df['age'] * 1.2 + df['experience_years'] * 2.0\n
dept_boost = df['department'].map({'sales': 20, 'tech': 35, 'hr': 10, 'ops': 15})\n
city_boost = df['city'].map({'stockholm': 25, 'goteborg': 15, 'malmo': 10})\n
noise = rng.normal(0, 18, size=n)\n
df['monthly_spend'] = (base + dept_boost + city_boost + noise).round(2)\n
\n
# Introducera lite saknade varden\n
df.loc[rng.choice(df.index, size=20, replace=False), 'experience_years'] = np.nan\n
df.loc[rng.choice(df.index, size=15, replace=False), 'city'] = None\n
\n
df.head()

In [ ]:
# Feature engineering\n
df['age_bucket'] = pd.cut(df['age'], bins=[17, 29, 44, 59, 100], labels=['18-29', '30-44', '45-59', '60+'])\n
df['exp_per_age'] = df['experience_years'] / df['age']\n
df['exp_per_age'] = df['exp_per_age'].replace([np.inf, -np.inf], np.nan)\n
\n
target = 'monthly_spend'\n
X = df.drop(columns=[target])\n
y = df[target]

In [ ]:
num_cols = X.select_dtypes(include=['number']).columns.tolist()\n
cat_cols = X.select_dtypes(exclude=['number']).columns.tolist()\n
\n
num_pipeline = Pipeline(steps=[\n
    ('imputer', SimpleImputer(strategy='median')),\n
    ('scaler', StandardScaler()),\n
])\n
\n
cat_pipeline = Pipeline(steps=[\n
    ('imputer', SimpleImputer(strategy='most_frequent')),\n
    ('onehot', OneHotEncoder(handle_unknown='ignore')),\n
])\n
\n
preprocess = ColumnTransformer(transformers=[\n
    ('num', num_pipeline, num_cols),\n
    ('cat', cat_pipeline, cat_cols),\n
])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(\n
    X, y, test_size=0.2, random_state=42\n
)\n
\n
model = RandomForestRegressor(n_estimators=250, random_state=42)\n
\n
pipe = Pipeline(steps=[\n
    ('preprocess', preprocess),\n
    ('model', model),\n
])\n
\n
pipe.fit(X_train, y_train)

In [ ]:
pred = pipe.predict(X_test)\n
\n
mae = mean_absolute_error(y_test, pred)\n
r2 = r2_score(y_test, pred)\n
\n
results = pd.DataFrame({'metric': ['MAE', 'R2'], 'value': [mae, r2]})\n
results

In [ ]:
# Exportera prediktioner\n
pred_df = X_test.copy()\n
pred_df['actual'] = y_test.values\n
pred_df['predicted'] = pred\n
\n
out_path = processed_dir / 'model_predictions.csv'\n
pred_df.to_csv(out_path, index=False)\n
out_path